<a href="https://colab.research.google.com/github/yy1010yy/data-science-toolkit/blob/main/Tutorial6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 0. Import the libraries and upload the two CSV files

In [2]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_salary = pd.read_csv('/content/drive/MyDrive/2026-27-S1-Intro to data science/datasets/salary/salary.csv')
df_xtra = pd.read_csv('/content/drive/MyDrive/2026-27-S1-Intro to data science/datasets/salary/xtra_info dataset.csv')

# print the number of rows  and columns in each table
print("Salary table size: ", df_salary.shape)
print("Etra table size: ", df_xtra.shape)

# Show a small preview of each dataset
display(df_salary.head(3))
display(df_xtra.head(3))

Salary table size:  (28180, 19)
Etra table size:  (13404, 4)


,Unnamed: 0,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,0,2021-04-27 11:02:09.743,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,55000,0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,1,2021-04-27 11:02:21.562,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,54600,4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,2,2021-04-27 11:02:38.125,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,34000,NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


,USERID,CGPA,Salary_month_USD,Special Bonus_GBP
0,8683,2.990627,4583.333333,2437.0
1,3880,2.288131,5166.666667,740.0
2,8898,3.903324,5000.000000,3806.0


### 0.1 Shorten long column Names using column numbers

In [3]:
# Build a rename dictionary using COLUMN POSITIONS (not long names).
short_names = {
    df_salary.columns[0]: "USERID",
    df_salary.columns[6]: "Annual_Salary",
    df_salary.columns[7]: "Extra_Compensation",
    df_salary.columns[8]: "Currency",
    df_salary.columns[11]: "Country",
    df_salary.columns[13]: "City"
}

# Apply the shorter names to a fresh copy of our salary data
df_salary = df_salary.rename(columns=short_names)

# Make sure IDs and numerical data are stored as numbers.
df_salary["USERID"] = pd.to_numeric(df_salary["USERID"], errors="coerce")
df_salary['Annual_Salary'] = pd.to_numeric(
    df_salary["Annual_Salary"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)
df_xtra["USERID"] = pd.to_numeric(df_xtra["USERID"], errors="coerce")

# Display only the columns we need for this class
display(df_salary[["USERID", "Annual_Salary", "Currency", "Country", "City"]].head())

,USERID,Annual_Salary,Currency,Country,City
0,0,55000,USD,United States,Boston
1,1,54600,GBP,United Kingdom,Cambridge
2,2,34000,USD,US,Chattanooga
3,3,62000,USD,USA,Milwaukee
4,4,60000,USD,US,Greenville


# 1. Data Integration

**Question:** Should we use `merge` or `concat`?

**Answer:** `pd.merge()` because both files have a shared `USERID`. `concat()` normally stacks tables beside each other or on top of each other; it doesn't automatically match people by ID.






### 1.1 Check whether the extra file contains repeated IDs

A `merge` needs a clear relationship between IDs. If the extra table has repeated IDs, one salary record could be copied into multiple merged rows by accident. Since the tutorial doesn't tell us which repeated record is correct, we keep the first record only for a demonstration. In real projects, we would ask where the duplicates came from before choosing one.

In [4]:
# Count the total number of extra rows and how many IDs are different
print("Extra rows: ", len(df_xtra))
print("Unique extra USERIDs: ", df_xtra["USERID"].nunique())

# Count all rows that repeat an earlier USERID.
print("Repeated USERID rows: ", df_xtra.duplicated(subset="USERID").sum())

# This is a teachng only decision: keep the FIRST row per USERID
df_xtra_unique = df_xtra.drop_duplicates(subset="USERID", keep="first").copy();

# Check that every USERID now occurs at most once.
print("After removing repeated IDs: ", len(df_xtra_unique))
assert not df_xtra_unique["USERID"].duplicated().any()

Extra rows:  13404
Unique extra USERIDs:  8442
Repeated USERID rows:  4962
After removing repeated IDs:  8442


### 1.1 Answer: Merge the two DataFrames

`how="left"` means keep all salary records, even when there is no matching extra-information record. `validate="one_to_one"` helps catch accidental duplicate matches.

In [5]:
# Merge extra columns onto the original salary rows using USERID.
df_integrated = pd.merge(
    df_salary,
    df_xtra_unique,
    on="USERID",
    how="left",
    validate="one_to_one"
)

# A safe left merge should not create extra salary rows.
print('Salary rows before merge:', len(df_salary))
print('Salary rows after merge:', len(df_integrated))

# We want to check how many rows received extra info
# to do this you can check how many are not null
print("Rows with extra information:", df_integrated["CGPA"].notna().sum())

# assert checks if the condition is true
# We want to check if both have the same num of rows to make sure no salary rows were added or lost
assert len(df_integrated) == len(df_salary)

# Look at the merged columns
display(df_integrated[["USERID", "Annual_Salary", "Currency", "CGPA", "Salary_month_USD", "Special Bonus_GBP"]].head(8))

Salary rows before merge: 28180
Salary rows after merge: 28180
Rows with extra information: 8442


,USERID,Annual_Salary,Currency,CGPA,Salary_month_USD,Special Bonus_GBP
0,0,55000,USD,NaN,NaN,NaN
1,1,54600,GBP,2.695615,4250.000000,125.0
2,2,34000,USD,NaN,NaN,NaN
3,3,62000,USD,NaN,NaN,NaN
4,4,60000,USD,2.689383,12500.000000,1133.0
5,5,62000,USD,NaN,NaN,NaN
6,6,33000,USD,3.541300,6333.333333,878.0
7,7,50000,USD,NaN,NaN,NaN


### Check whether the monthly salary from the extra file is actually derivable

The tutorial asks us to remove a **derivable** column: something we can calculate from existing values. For example, a monthly salary could be calculated from an annual salary **if they are in the same currency and belong to the same record**.
Our supplied extra file contains values that do not consistently match that calculation, so we should not casually delete `Salary_month_USD` by saying it is redundant. First, we compare it with USD survey records,

In [6]:
# Compare monthly salary figures only when original salary currency is USD.
# Using & means if both conditions are true then selected these three columns
comparison = df_integrated.loc[
    (df_integrated["Currency"] == "USD") &
    df_integrated["Salary_month_USD"].notna(),
    ["USERID", "Annual_Salary", "Salary_month_USD"]
].copy()

# Calculate what a monthly salary would be from the annual USD salary.
comparison["Calculated_Monthly_USD"] = comparison["Annual_Salary"] / 12

# Check roughly how many supplied figures agree (allow tiny rounding differnences).
rough_match = (
    comparison["Salary_month_USD"] - comparison["Calculated_Monthly_USD"]
).abs() < 0.01

print("Number of comparable records:", len(comparison))
print("Matching monthly amounts:", rough_match.sum())
print("Non-matching monthly amounts:", (~rough_match).sum())

# Show a few examples to make the issue visible
display(comparison.head())

Number of comparable records: 7063
Matching monthly amounts: 55
Non-matching monthly amounts: 7008


,USERID,Annual_Salary,Salary_month_USD,Calculated_Monthly_USD
4,4,60000,12500.000000,5000.000000
6,6,33000,6333.333333,2750.000000
9,9,45000,12500.000000,3750.000000
10,10,47500,4583.333333,3958.333333
12,12,100000,12083.333330,8333.333333


### 1.2 Drop derivable data

**Answer:** Make an obviously derived column (`Annual_Salary_Div_12`) and remove it again using `drop()`. This teaches the exact method without making unsupported assumptions about the supplied `Salary_month_USD`. Note that the derived result has the **same currency as the source annual salary**, not necessarily USD.

In [7]:
# A genuinely derivable feature is yearly salary divided by 12
df_integrated["Annual_Salary_Div_12"] = df_integrated["Annual_Salary"] / 12

# Remove that calculated column because we can recreate it whenever needed.
df_integrated = df_integrated.drop(columns=["Annual_Salary_Div_12"])

# Confirm it has been removed
print("Derived column removed:", "Annual_Salary_Div_12" not in df_integrated.columns)

Derived column removed: True


### 1.3 Convert `Special_Bonus_GBP` to USD

**Answer:** `USD bonus GBP bonus x GBP TO_USD`

For classroom purposes, we assume **1GBP = 1.3U** . We create the hew name requested by the tutorial, and then drop the original
GBP column to avoid confusion.

In [8]:
# Classroom exchange rate assumption, not a live exchange rate
GBP_TO_USD = 1.30

# Create a new column containing the converted bonus In USD.
df_integrated["Special_Bonus_USD"] = (
  pd.to_numeric(df_integrated["Special Bonus_GBP"], errors="coerce")
  * GBP_TO_USD
)

# errors coerce means "If a value cannot be converted into a number, replace it with NaN (missing value) instead of showing an error

# Drop the old GBP bonus column now that we have a USD version.
df_integrated = df_integrated.drop(columns=["Special Bonus_GBP"])

#Display the converted bonus for a few matching records,
display(df_integrated.loc[
    df_integrated["Special_Bonus_USD"].notna(),
     ["USERID", "Special_Bonus_USD"]
].head())

,USERID,Special_Bonus_USD
1,1,162.5
4,4,1472.9
6,6,1141.4
9,9,6277.7
10,10,1016.6


# 2. Data Transformstion -- Normalization



Machine-learning methods sometimes work better when numeric inputs have similar scales.

*   **Min-Max** puts values between 0 and 1 (for the values used when fitting).
*   **Z-score/StandardScaler** changes values so the fitted data have mean near O and standard deviation near 1.



**Important:** We should not mix salaries from different currencies. For this exercise, we use rows whose original salary currency is `USD`. We also exclude missing or nonpositive salaries. Extreme values can make Min-Max scaling less informative; we leave them visible for learning rather than silently removing them.

In [10]:
# Select only salaries reported in US dollars.
df_usd = df_integrated.loc[df_integrated["Currency"] == "USD"].copy()

# Keep only sensible, numeric, positive salaries
df_usd = df_usd.loc[
    df_usd["Annual_Salary"].notna() & (df_usd["Annual_Salary"] > 0)
].copy()

# Print the size and sample annual salary values
print("USD salary records:", len(df_usd))
display(df_usd[["Annual_Salary"]].head())

USD salary records: 23441


,Annual_Salary
0,55000
2,34000
3,62000
4,60000
5,62000


### 2.1 Min-Max normalization

Formula: `(value minimum) / (maximum minimum)`.

**Example:** If the minimum is 10 and maximum is 30, then 20 becomes `(20-10)/(30-18)=0.5.`

In [15]:
# Create a MinMaxScaler, which learns the smalllest and biggest salary.
minmax = MinMaxScaler()

# fit_transform learns the range and scales the column to 0-1
df_usd["Salary_MinMax"] = minmax.fit_transform(df_usd[["Annual_Salary"]]).ravel()
# ravel converts the result from a two-dimensional array into a one-dimensional array
# from this [[10.0], [0.5], [1.0]] to this [0.0, 8.5, 1.0]

# Compare the original salary with its scaled value.
display(df_usd[["Annual_Salary", "Salary_MinMax"]].head(10))
print("Minimum scaled salary:", df_usd ["Salary_MinMax"].min())
print("Maximun scaled salary:", df_usd ["Salary_MinMax"].max())

,Annual_Salary,Salary_MinMax
0,55000,0.000539
2,34000,0.000333
3,62000,0.000608
4,60000,0.000588
5,62000,0.000608
6,33000,0.000324
7,50000,0.000490
8,112000,0.001098
9,45000,0.000441
10,47500,0.000466


Minimum scaled salary: 0.0
Maximun scaled salary: 1.0


### 2.2 Z-score standardization

Formula: `(value mean) / standard deviation`

A 2-score of **0** means close to the average; **positive** means above average; **negative** means below average. Unlike Min-Max, Z-scores are not restricted to 0-1.

In [16]:
# Create a StandardScaler, which learns the mean and standard deviation.
zscore_scaler = StandardScaler()

# Fit the scaler using the annual USD salaries and transform them.
df_usd["Salary_Zscore"] = zscore_scaler.fit_transform(df_usd[["Annual_Salary"]]).ravel()

# Compare the original salary with both scaling results.
display(df_usd[["Annual_Salary", "Salary_MinMax", "Salary_Zscore"]].head(10))

# Calculate the average of the standardized salaries
print("Mean:", df_usd["Salary_Zscore"].mean())

# Calculate how spread out the standardized salaries are
print("Standard deviation:", df_usd["Salary_Zscore"].std())

,Annual_Salary,Salary_MinMax,Salary_Zscore
0,55000,0.000539,-0.058724
2,34000,0.000333,-0.086545
3,62000,0.000608,-0.049450
4,60000,0.000588,-0.052100
5,62000,0.000608,-0.049450
6,33000,0.000324,-0.087870
7,50000,0.000490,-0.065348
8,112000,0.001098,0.016792
9,45000,0.000441,-0.071972
10,47500,0.000466,-0.068660


Mean: -9.699828311216759e-18
Standard deviation: 1.0000213308305577


# 3. Data Reduction -- Aggregation and Grouping

The tutorial asks us to **group salaries by city**, calculate summaries, and sort the cities from highest to lowest salary.

We include **Country** alongside City because different countries can have cities with the same name. These results refer only to the **USD-reported survey records**, not to all workers in a city. A city with one very high salary can have a misleading average.

In [17]:
# Remove rows with missing/empty city names.
# Remove rows with missing city names
df_city = df_usd.dropna(subset=["City"])

# groupby collects records in the same country/city combination.
# agg calculates different summary statistics for each group.

# This code groups employees by their country and city, then calculates salary statistics for each location
# It puts rows belonging to the same country and city into groups.
# For example, all three New York salaries are grouped together.
# as_index=False simply keeps Country and City as normal columns in the resulting table.

city_summary = df_city.groupby(["Country", "City"], as_index=False).agg(
    Number_of_Records=("Annual_Salary", "count"), # How many reported salaries
    Mean_Salary_USD=("Annual_Salary", "mean"), # Average annual salary
    Median_Salary_USD=("Annual_Salary", "median"), # Median annual salary
    Minimum_Salary_USD=("Annual_Salary", "min"), # Minimum annual salary
    Maximum_Salary_USD=("Annual_Salary", "max"), # Maximum annual salary
)

# Display a sample of the grouped results.
display(city_summary.head(10))

,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
0,U.S.,indianapolis,1,42500.0,42500.0,42500,42500
1,US,Philadelphia,1,86000.0,86000.0,86000,86000
2,United States,South Bend,1,69000.0,69000.0,69000,69000
3,Africa,Africa,1,90000.0,90000.0,90000,90000
4,America,Ann Arbor,1,34000.0,34000.0,34000,34000
5,America,Atlanta,1,45000.0,45000.0,45000,45000
6,America,Boston,1,194000.0,194000.0,194000,194000
7,America,Chicago,1,70.0,70.0,70,70
8,America,Columbia,1,75000.0,75000.0,75000,75000
9,America,Houston,1,20488.0,20488.0,20488,20488


### 3.2 Sort cities from highest mean salary to lowest

`ascending=False` means **highest first**

In [18]:
# Sort all groups by average annual salary, highest to lowest
all_cities_sorted = city_summary.sort_values(
    by="Mean_Salary_USD",
    ascending=False
)

# Show the ten cities with the highest sample mean salary.
print("Top 10 city/country groups (all sizes):")
display(all_cities_sorted.head(10))

Top 10 city/country groups (all sizes):


,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
62,Colombia,Bogota,2,51012500.0,51012500.0,25000,102000000
64,Colombia,Cali,2,26106000.0,26106000.0,12000,52200000
6924,usa,Rice,1,10000000.0,10000000.0,10000000,10000000
6609,Usa,Pinebrook,1,5000044.0,5000044.0,5000044,5000044
59,China,Hong Kong,1,5000000.0,5000000.0,5000000,5000000
2524,USA,Newyork,1,2600000.0,2600000.0,2600000,2600000
175,Taiwan,Milwaukee,1,2000000.0,2000000.0,2000000,2000000
102,India,Banglore,1,1000000.0,1000000.0,1000000,1000000
3639,United Stated,Spokane,1,832000.0,832000.0,832000,832000
430,U.S.,Salina,1,734400.0,734400.0,734400,734400
